# MLP Contribution Profiling

Analyze how MLPs transform the residual stream: contributions, logit effects,
position profiles, layer comparison, and neuron rankings.

## Why This Matters

MLP contribution provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_contribution_profiling import (
    mlp_residual_contribution, mlp_logit_effect,
    mlp_position_profile, mlp_layer_comparison,
    mlp_neuron_contribution_ranking,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Residual Contribution

How much does each MLP layer contribute to the residual stream?

In [ ]:
result = mlp_residual_contribution(model, tokens)
print(f"Constructive: {result['n_constructive']}/{len(result['per_layer'])}\n")
for p in result['per_layer']:
    sign = '+' if p['is_constructive'] else '-'
    bar = '█' * int(p['fraction_of_residual'] * 20)
    print(f"  Layer {p['layer']}: mlp={p['mlp_norm']:.4f}, "
          f"frac={p['fraction_of_residual']:.1%}, "
          f"align={p['alignment_to_residual']:+.4f} [{sign}] {bar}")

## Logit Effect

What tokens does each MLP promote/suppress?

In [ ]:
result = mlp_logit_effect(model, tokens, position=-1)
for p in result['per_layer']:
    promoted = ', '.join(f"{t['token']}:{t['logit']:+.3f}" for t in p['top_promoted'][:3])
    suppressed = ', '.join(f"{t['token']}:{t['logit']:+.3f}" for t in p['top_suppressed'][:3])
    print(f"  Layer {p['layer']} (norm={p['logit_norm']:.4f}):")
    print(f"    promotes: {promoted}")
    print(f"    suppresses: {suppressed}")

## Position Profile

Does the MLP treat all positions equally, or specialize?

In [ ]:
for layer in range(4):
    result = mlp_position_profile(model, tokens, layer=layer)
    unif = 'UNIFORM' if result['is_position_uniform'] else 'variable'
    print(f"Layer {layer} (cv={result['norm_cv']:.4f}) [{unif}]:")
    for p in result['per_position']:
        aligned = '✓' if p['is_aligned'] else '✗'
        print(f"  Pos {p['position']}: norm={p['norm']:.4f}, "
              f"align={p['alignment_to_mean']:+.4f} [{aligned}]")
    print()

## Layer Comparison

Do MLP layers write in similar or unique directions?

In [ ]:
result = mlp_layer_comparison(model, tokens)
print(f"Unique layers: {result['n_unique']}\n")
for p in result['per_layer']:
    uniq = ' [UNIQUE]' if p['is_unique'] else ''
    print(f"  Layer {p['layer']}: norm={p['mean_norm']:.4f}, "
          f"sim_to_others={p['mean_similarity_to_others']:.4f}{uniq}")

## Neuron Contribution Ranking

Which neurons contribute most at a given position?

In [ ]:
for layer in range(4):
    result = mlp_neuron_contribution_ranking(model, tokens, layer=layer, top_k=5)
    print(f"Layer {layer} ({result['n_active']}/{result['total_neurons']} active):")
    for n in result['per_neuron']:
        active = '✓' if n['is_active'] else '✗'
        print(f"  Neuron {n['neuron']:3d}: act={n['activation']:+.4f}, "
              f"out_norm={n['output_norm']:.4f}, "
              f"contrib={n['contribution']:.4f} [{active}]")
    print()